# Tome C3, Chapter 8: Time and Space (Python companion)

Exercises first, solutions at the end. We measure memory in bits, run Savitch's recursion and watch its time explode while its stack stays tiny, certify that a vertex is *not* reachable by inductive counting, build a diagonaliser and an oracle that defeats a whole family of fast machines, and finish with two problems that live in small space: 2-SAT and reachability in undirected graphs by random walk.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, math, random, time
from collections import deque
import matplotlib.pyplot as plt
random.seed(8)

## Exercises

**P8.1 (★) Pointers are logarithmic.** Decide palindromes and $\{a^nb^nc^n\}$ with programs whose only memory is a few integer registers (Exercise 8.1). Record the largest number of bits the registers ever hold, and plot it against $n$. With that space, how many configurations can the machine have (equation (8.1)), and why does it follow that it halts in polynomial time?

**P8.2 (★★) Savitch's recursion.** Implement $\mathrm{REACH}(u,v,t)$ of Theorem 8.7 on directed graphs, counting calls and the deepest stack. On paths with $n$ vertices compare the number of calls with $n^{\log_2 n}$, and the stack size with $\log_2 n$ frames of $O(\log n)$ bits. What does breadth-first search use instead?

**P8.3 (★★) Certifying an absence.** Simulate the nondeterministic machine of Theorem 8.8: a *guesser* supplies the choices, a *verifier* with a few counters checks them. Run it on the graph of Figure 8.3 and recover $c_0,\dots,c_4$ and the certificate that $t$ is not reachable. Then let the guesser lie, at random, and check that the verifier is never fooled.

**P8.4 (★) A diagonaliser.** Model machines as Python generators that yield once per step. Build $D$, which on input $i$ runs machine $i$ on $i$ for $i^2$ steps and answers the opposite (Theorem 8.5). Check that $D$ disagrees with every machine that runs within the budget, and see why each machine must appear in the list infinitely often.

**P8.5 (★★) 2-SAT in small space.** Decide 2-SAT by reachability in the implication graph (Exercise 8.7) and compare with brute force on random formulas. Then plot the probability that a random 2-CNF formula with $n$ variables and $\alpha n$ clauses is satisfiable, as a function of $\alpha$.

**P8.6 (★) Odd cycles by reachability.** A graph is bipartite if and only if no vertex $v$ reaches $(v,1)$ from $(v,0)$ in the doubled graph of Exercise 8.8. Implement this test and check it against two-colouring by breadth-first search on random graphs.

**P8.7 (★★) An oracle against fast machines.** Build, stage by stage, the oracle $B$ of Theorem 8.9 against a family of deterministic oracle algorithms that make at most $n^2$ queries on $1^n$, each trying to decide whether $B$ has a word of length $n$. Verify that every one of them is wrong at its stage, and that an algorithm allowed $2^n$ queries is right.

**P8.8 (★) Walking instead of remembering.** In an undirected graph, a random walk needs only the current vertex and a step counter. Measure the time it takes to visit every vertex (the cover time) on paths, cycles, complete graphs and lollipops, and compare with the bound $2m(n-1)$ of Aleliunas, Karp, Lipton, Lovász and Rackoff (1979).

## Solutions

### P8.1

In [ ]:
class Registers:
    # integer registers; we record the largest number of bits ever stored
    def __init__(self):
        self.r, self.peak = {}, 0
    def __setitem__(self, k, v):
        self.r[k] = v
        self.peak = max(self.peak, sum(max(1, x.bit_length()) for x in self.r.values()))
    def __getitem__(self, k):
        return self.r[k]

def palindrome(w, R):
    R['n'] = len(w)
    R['i'] = 0
    while R['i'] < R['n'] // 2:
        # reading w[j] stands for moving the input head to position j
        if w[R['i']] != w[R['n'] - 1 - R['i']]:
            return False
        R['i'] = R['i'] + 1
    return True

def abc(w, R):
    R['state'] = 0                       # finite automaton for a*b*c*
    for ch in w:
        s = R['state']
        if ch not in 'abc' or 'abc'.index(ch) < s:
            return False
        R['state'] = 'abc'.index(ch)
    for x in 'abc':
        R['count_' + x] = 0
        for ch in w:
            if ch == x:
                R['count_' + x] = R['count_' + x] + 1
    return R['count_a'] == R['count_b'] == R['count_c']

ns, peaks_p, peaks_abc = [], [], []
for k in range(2, 21):
    n = 2 ** k // 4 * 3
    w = ''.join(random.choice('ab') for _ in range(n // 2))
    w = w + w[::-1]
    R = Registers(); assert palindrome(w, R); peaks_p.append(R.peak)
    m = n // 3
    R = Registers(); assert abc('a' * m + 'b' * m + 'c' * m, R); peaks_abc.append(R.peak)
    ns.append(n)
for n, p, q in list(zip(ns, peaks_p, peaks_abc))[::4]:
    print(f"n = {n:8d}: palindrome {p:3d} bits, a^n b^n c^n {q:3d} bits, log2 n = {math.log2(n):5.1f}")
plt.figure(figsize=(6, 3.5))
plt.semilogx(ns, peaks_p, 'o-', label='palindromes')
plt.semilogx(ns, peaks_abc, 's-', label=r'$a^nb^nc^n$')
plt.semilogx(ns, [2 * math.log2(n) for n in ns], 'k--', label=r'$2\log_2 n$')
plt.xlabel('input length n'); plt.ylabel('peak bits in registers'); plt.legend(); plt.tight_layout(); plt.show()

Both programs use a handful of registers holding numbers at most $n$: about $2\log_2 n$ bits for palindromes, a little more for $a^nb^nc^n$ (three counters and a state). That is the whole content of "logarithmic space": a constant number of pointers into the input.

With $s=c\log_2 n$ bits of work memory, a configuration is a state, a head position on the input ($n+2$ choices), and the content of the memory: at most $|Q|(n+2)2^{s}=|Q|(n+2)n^{c}$ configurations, a polynomial. A deterministic machine that halts never repeats a configuration (otherwise it would loop forever), so it halts within that many steps: $\mathsf L\subseteq\mathsf P$, as in Proposition 8.2.

### P8.2

In [ ]:
def savitch(G, u, v, t, stats, depth=1):
    stats['calls'] += 1
    stats['depth'] = max(stats['depth'], depth)
    if t <= 1:
        return u == v or v in G[u]
    for m in G:                          # try every midpoint
        if savitch(G, u, m, t // 2, stats, depth + 1) and savitch(G, m, v, t // 2, stats, depth + 1):
            return True
    return False

def path_graph(n):
    return {i: ({i + 1} if i + 1 < n else set()) for i in range(n)}

def bfs_reach(G, s):
    seen, q = {s}, deque([s])
    while q:
        x = q.popleft()
        for y in G[x]:
            if y not in seen:
                seen.add(y); q.append(y)
    return seen

rows = []
for n in [2, 4, 6, 8, 10, 12, 14, 16]:
    G = path_graph(n)
    t = 1 << max(0, (n - 1).bit_length())      # a power of 2 at least n-1
    st = {'calls': 0, 'depth': 0}
    t0 = time.perf_counter()
    ok = savitch(G, 0, n - 1, t, st)
    dt = time.perf_counter() - t0
    assert ok == (n - 1 in bfs_reach(G, 0))
    rows.append((n, t, st['calls'], st['depth'], dt))
    print(f"n = {n:2d}, t = {t:2d}: {st['calls']:8d} calls, deepest stack {st['depth']} frames, "
          f"n^log2(n) = {n ** math.log2(n):10.0f}, {dt * 1000:7.1f} ms")
plt.figure(figsize=(6, 3.5))
plt.semilogy([r[0] for r in rows], [r[2] for r in rows], 'o-', label='calls of REACH')
plt.semilogy([r[0] for r in rows], [r[0] ** math.log2(r[0]) for r in rows], 'k--', label=r'$n^{\log_2 n}$')
plt.xlabel('vertices n (path graph)'); plt.legend(); plt.tight_layout(); plt.show()

The stack never holds more than $\log_2 t+1$ frames, and each frame stores $u$, $v$, $t$ and the current midpoint: $O(\log n)$ bits, so $O(\log^2 n)$ bits in all. The number of calls, on the other hand, grows like $n^{\log_2 n}$: each level of the recursion tries up to $n$ midpoints and makes two calls for each, through $\log_2 n$ levels. (The count follows $n^{\log_2 n}$ closely; it cannot be compared exactly, since $t$ is rounded up to a power of $2$ and a failed first half skips the second call.) That is Savitch's trade: almost all the time in the world for very little memory.

Breadth-first search is fast, linear in the size of the graph, but its `seen` set and queue can hold every vertex: $n$ bits at least. Nobody knows whether reachability in directed graphs can be solved in polynomial time *and* $O(\log^2 n)$ space at the same time; the best known algorithms trade one against the other.

### P8.3

In [ ]:
FIG = {'s': ['a', 'b'], 'a': ['c'], 'b': ['d'], 'c': ['e'], 'd': ['b', 'e'],
       'e': ['f'], 'f': ['c'], 't': ['d']}
FIG = {x: set(ys) for x, ys in FIG.items()}

def distances(G, s):
    d, q = {s: 0}, deque([s])
    while q:
        x = q.popleft()
        for y in G[x]:
            if y not in d:
                d[y] = d[x] + 1; q.append(y)
    return d

class Reject(Exception):
    pass

def check_path(G, s, path, v, i):
    # the verifier reads a guessed path one vertex at a time: O(log n) bits
    if path[0] != s or path[-1] != v or len(path) - 1 > i:
        raise Reject
    for x, y in zip(path, path[1:]):
        if y not in G[x]:
            raise Reject

class HonestGuesser:
    def __init__(self, G, s):
        self.G, self.s, self.d = G, s, distances(G, s)
    def in_R(self, u, i):                # guess: is u at distance <= i?  if so, a path
        if u in self.d and self.d[u] <= i:
            path, x = [u], u
            while x != self.s:
                x = next(p for p in self.G if x in self.G[p] and self.d.get(p, -9) == self.d[x] - 1)
                path.append(x)
            return path[::-1]
        return None

class LyingGuesser(HonestGuesser):
    def __init__(self, G, s, rate, rng):
        super().__init__(G, s); self.rate, self.rng = rate, rng
    def in_R(self, u, i):
        p = super().in_R(u, i)
        if p is not None and self.rng.random() < self.rate:
            return None                  # hide a reachable vertex
        if p is None and self.rng.random() < self.rate:
            return [u]                   # claim a fake path
        return p

def next_count(G, s, i, c_i, guesser):
    # computes c_{i+1} from c_i; every claim is checked
    count = 0
    for v in G:
        seen, in_next = 0, False
        for u in G:
            path = guesser.in_R(u, i)
            if path is None:
                continue
            check_path(G, s, path, u, i)
            seen += 1
            if u == v or v in G[u]:
                in_next = True
        if seen != c_i:                  # the guesser hid a member of R_i
            raise Reject
        count += in_next
    return count

def certify_unreachable(G, s, t, guesser):
    n, c = len(G), [1]
    for i in range(n - 1):
        c.append(next_count(G, s, i, c[-1], guesser))
    seen = 0
    for u in G:
        path = guesser.in_R(u, n - 1)
        if path is None:
            continue
        check_path(G, s, path, u, n - 1)
        if u == t:
            return c, False              # t was shown to be reachable
        seen += 1
    if seen != c[-1]:
        raise Reject
    return c, True

c, unreachable = certify_unreachable(FIG, 's', 't', HonestGuesser(FIG, 's'))
print("counts c_i:", c, "| t certified unreachable:", unreachable)

rng = random.Random(1)
fooled = caught = 0
for trial in range(300):
    try:
        res = certify_unreachable(FIG, 's', 't', LyingGuesser(FIG, 's', 0.1, rng))
        if res[0] != c:
            fooled += 1
    except Reject:
        caught += 1
print(f"lying guesser: {caught} runs rejected, {300 - caught} accepted, of which {fooled} with wrong counts")

The honest run reproduces Figure 8.3: $c_0,\dots,c_7=1,3,5,6,7,7,7,7$ (the count stabilises at $7$), and $t$ is certified unreachable: the verifier saw seven vertices with valid paths, none of them $t$, and seven is the right number.

A liar can try two things. A fake path is caught by `check_path`. Hiding a reachable vertex makes the number of vertices seen smaller than $c_i$, which the verifier knows, so the run is rejected. Every lying run is rejected, and no accepted run ever carries a wrong count: a nondeterministic machine may reject on a bad guess, but it can never accept wrongly. That is why the counts, computed one layer at a time, can be trusted, and why $\mathsf{NL}=\mathsf{coNL}$. The verifier's memory is a few counters and two or three vertex names: $O(\log n)$ bits.

### P8.4

In [ ]:
# machines: generator functions; each yield is one step; the return value is the answer
def m_const_yes(x):
    yield
    return True
def m_parity(x):
    for _ in range(x.bit_length()):
        yield
    return x % 2 == 0
def m_slow_square(x):                # x^2/2 + 20 steps, then says "x is a square"
    for _ in range(x * x // 2 + 20):
        yield
    return math.isqrt(x) ** 2 == x
def m_cubic(x):                      # x^3 steps: beyond the budget
    for _ in range(x ** 3):
        yield
    return False
def m_loop(x):
    while True:
        yield

MACHINES = [m_const_yes, m_parity, m_slow_square, m_cubic, m_loop]

def machine(i):                      # every machine has infinitely many codes
    return MACHINES[i % len(MACHINES)]

def run(M, x, budget):
    g = M(x)
    for _ in range(budget):
        try:
            next(g)
        except StopIteration as e:
            return e.value
    return None                      # out of time

def D(i):
    ans = run(machine(i), i, i * i)
    return True if ans is None else not ans

for i in range(1, 26):
    M = machine(i)
    a = run(M, i, 10 ** 6)
    within = run(M, i, i * i) is not None
    print(f"i = {i:2d}  {M.__name__:14s} answers {str(a):5s}  "
          f"within i^2 steps: {str(within):5s}  D(i) = {D(i)!s:5s}  differs: {D(i) != a}")

Whenever machine $i$ finishes within $i^2$ steps on input $i$, $D$ answers the opposite, so $D$ differs from it. A machine that is too slow on its own code (here `m_slow_square` for $i<7$, `m_cubic` and `m_loop` always) escapes on that code, but this is no loss: a machine that runs in time $O(n^{1.5})$, or $n^2/2+20$ like `m_slow_square`, is within budget for all large $i$, and since it has infinitely many codes, $D$ catches it on one of them. That is why the proof of Theorem 8.5 lets every machine appear infinitely often. The two loose ends of the real proof, the clock and the cost of simulation, are what the hypothesis $f^2=o(g)$ pays for; `m_cubic` is a machine that $D$ is not built to defeat (it differs from it here only by luck, because `m_cubic` always says no), rightly so, since $D$ itself runs in time about $n^2$.

### P8.5

In [ ]:
def random_2cnf(n, m, rng):
    cl = []
    for _ in range(m):
        a, b = rng.sample(range(1, n + 1), 2)
        cl.append((a * rng.choice((1, -1)), b * rng.choice((1, -1))))
    return cl

def implication_graph(n, cl):
    G = {l: set() for v in range(1, n + 1) for l in (v, -v)}
    for a, b in cl:
        G[-a].add(b); G[-b].add(a)
    return G

def sat_by_reachability(n, cl):
    G = implication_graph(n, cl)
    for x in range(1, n + 1):
        if -x in bfs_reach(G, x) and x in bfs_reach(G, -x):
            return False
    return True

def sat_brute(n, cl):
    for bits in itertools.product((False, True), repeat=n):
        val = lambda l: bits[abs(l) - 1] if l > 0 else not bits[abs(l) - 1]
        if all(val(a) or val(b) for a, b in cl):
            return True
    return False

rng = random.Random(3)
for trial in range(300):
    n = rng.randint(2, 10); m = rng.randint(1, 3 * n)
    cl = random_2cnf(n, m, rng)
    assert sat_by_reachability(n, cl) == sat_brute(n, cl)
print("reachability test agrees with brute force on 300 random formulas")

alphas = [0.2 * k for k in range(1, 16)]
plt.figure(figsize=(6, 3.5))
for n in [20, 80, 320]:
    probs = []
    for a in alphas:
        trials = 60
        probs.append(sum(sat_by_reachability(n, random_2cnf(n, int(a * n), rng)) for _ in range(trials)) / trials)
    plt.plot(alphas, probs, 'o-', ms=3, label=f'n = {n}')
plt.axvline(1, color='k', ls='--', lw=0.8)
plt.xlabel(r'clauses per variable $\alpha$'); plt.ylabel('P(satisfiable)'); plt.legend(); plt.tight_layout(); plt.show()

The reachability criterion of Aspvall, Plass and Tarjan agrees with brute force everywhere. Each test "$\neg x$ reachable from $x$" is a reachability question, which a nondeterministic machine answers in logarithmic space, and the complement (both paths exist for some $x$) is what Exercise 8.7 puts in $\mathsf{NL}$; Theorem 8.8 does the rest.

The plot shows a *phase transition*: random 2-CNF formulas are almost surely satisfiable when $\alpha<1$ and almost surely unsatisfiable when $\alpha>1$, and the switch gets sharper as $n$ grows. This was proved by Chvátal and Reed (1992) and independently by Goerdt (1996); for 3-SAT the threshold is believed to be near $4.27$, and is still not known exactly.

### P8.6

In [ ]:
def random_graph(n, p, rng):
    G = {v: set() for v in range(n)}
    for u, v in itertools.combinations(range(n), 2):
        if rng.random() < p:
            G[u].add(v); G[v].add(u)
    return G

def bipartite_by_reachability(G):
    D = {(v, b): {(w, 1 - b) for w in G[v]} for v in G for b in (0, 1)}
    return all((v, 1) not in bfs_reach(D, (v, 0)) for v in G)

def bipartite_by_colouring(G):
    col = {}
    for s in G:
        if s in col:
            continue
        col[s], q = 0, deque([s])
        while q:
            x = q.popleft()
            for y in G[x]:
                if y not in col:
                    col[y] = 1 - col[x]; q.append(y)
                elif col[y] == col[x]:
                    return False
    return True

rng = random.Random(5)
agree, yes = 0, 0
for trial in range(400):
    n = rng.randint(2, 25)
    G = random_graph(n, rng.choice([0.05, 0.1, 0.2]), rng)
    a, b = bipartite_by_reachability(G), bipartite_by_colouring(G)
    agree += a == b; yes += a
print(f"agreement on {agree}/400 random graphs ({yes} bipartite)")

A walk from $(v,0)$ to $(v,1)$ in the doubled graph alternates layers, so it has odd length; it projects to a closed walk of odd length at $v$, which contains an odd cycle. Conversely an odd cycle through $v$ lifts to such a walk. So non-bipartiteness is "there exist $v$ and a path", which is in $\mathsf{NL}$, and bipartiteness is in $\mathsf{coNL}=\mathsf{NL}$. (Reingold's theorem even puts it in $\mathsf L$, since the doubled graph is undirected.)

### P8.7

In [ ]:
def words(n):
    return [''.join(b) for b in itertools.product('01', repeat=n)]

# deterministic oracle algorithms; on input n they may ask at most n^2 questions
def alg_first(n, ask):
    return any(ask(w) for w in words(n)[: n * n])
def alg_last(n, ask):
    return any(ask(w) for w in words(n)[-(n * n):])
def alg_sparse(n, ask):          # words with at most two 1s
    ws = [w for w in words(n) if w.count('1') <= 2][: n * n]
    return any(ask(w) for w in ws)
def alg_adaptive(n, ask):        # walks a path in the binary tree, guided by answers
    w = '0' * n
    for k in range(n * n):
        if ask(w):
            return True
        w = format((int(w, 2) * 5 + 3 + k) % 2 ** n, f'0{n}b')
    return False
def alg_random(n, ask):
    r = random.Random(n)
    return any(ask(format(r.randrange(2 ** n), f'0{n}b')) for _ in range(n * n))
def alg_says_no(n, ask):
    return False
def alg_says_yes(n, ask):
    return True
ALGS = [alg_first, alg_last, alg_sparse, alg_adaptive, alg_random, alg_says_no, alg_says_yes]

class Oracle:
    def __init__(self, B, decided, budget):
        self.B, self.decided, self.budget, self.asked = B, decided, budget, []
    def __call__(self, w):
        self.asked.append(w)
        if len(self.asked) > self.budget:
            raise RuntimeError("too many queries")
        if w not in self.decided:        # undecided words answer "no", and are now decided
            self.decided.add(w)
        return w in self.B

B, decided, stage_n, n = set(), set(), {}, 4
for A in ALGS:
    n += 1
    while 2 ** n <= n * n:
        n += 1
    O = Oracle(B, decided, n * n)
    says = A(n, O)
    if not says:                         # it says "no word of length n": put one in
        w = next(w for w in words(n) if w not in decided)
        B.add(w); decided.add(w)
    decided.update(words(n))             # length n is now settled
    stage_n[A.__name__] = n

for A in ALGS:
    n = stage_n[A.__name__]
    truth = any(w in B for w in words(n))
    says = A(n, lambda w: w in B)
    print(f"{A.__name__:13s} stage n = {n:2d}: B has a word of length n: {str(truth):5s}, algorithm says {str(says):5s}  -> wrong: {says != truth}")

n = 9
brute = lambda n, ask: any(ask(w) for w in words(n))
print(f"\nbrute force with 2^{n} = {2 ** n} queries at n = {n}: says {brute(n, lambda w: w in B)}, "
      f"truth {any(w in B for w in words(n))}")

At each stage the algorithm asks fewer than $2^n$ questions, all answered "no", so at least one word of length $n$ remains unasked. If it says "no word of length $n$", the stage puts an unasked word into $B$; if it says "yes", the stage leaves length $n$ empty. The answers it received stay true in the final $B$, so it behaves the same, and it is wrong. (The algorithms that ask hear only "no" and so all answer "no"; the stubborn `alg_says_yes` shows the other case, where length $n$ is left empty.) The language $\{1^n: B\text{ has a word of length }n\}$ is in $\mathsf{NP}^B$ (guess the word, ask once), yet no algorithm in the list decides it; the real proof runs through all polynomial-time oracle machines and gives $\mathsf P^B\ne\mathsf{NP}^B$. The brute-force algorithm, with exponentially many questions, is always right, which is the point: the oracle defeats speed, not cleverness.

### P8.8

In [ ]:
def cover_time(G, rng, start=0):
    seen, x, steps = {start}, start, 0
    nbrs = {v: list(G[v]) for v in G}
    while len(seen) < len(G):
        x = rng.choice(nbrs[x]); steps += 1
        seen.add(x)
    return steps

def path_u(n):  return {i: {j for j in (i - 1, i + 1) if 0 <= j < n} for i in range(n)}
def cycle(n):   return {i: {(i - 1) % n, (i + 1) % n} for i in range(n)}
def complete(n): return {i: set(range(n)) - {i} for i in range(n)}
def lollipop(n):                         # clique on n/2 vertices, then a path, walk starts in the clique
    k = n // 2
    G = {i: set(range(k)) - {i} for i in range(k)}
    for i in range(k, n):
        G[i] = {i - 1}
        G[i - 1].add(i)
    return G

rng = random.Random(9)
print(f"{'graph':10s} {'n':>4s} {'m':>6s} {'mean cover time':>16s} {'2m(n-1)':>10s} {'ratio':>6s}")
data = {}
for name, f in [('path', path_u), ('cycle', cycle), ('complete', complete), ('lollipop', lollipop)]:
    for n in [10, 20, 40, 80]:
        G = f(n)
        m = sum(len(v) for v in G.values()) // 2
        trials = 200 if name != 'lollipop' else 15
        ct = sum(cover_time(G, rng) for _ in range(trials)) / trials
        bound = 2 * m * (n - 1)
        data.setdefault(name, []).append((n, ct))
        if n in (20, 80):
            print(f"{name:10s} {n:4d} {m:6d} {ct:16.0f} {bound:10d} {ct / bound:6.3f}")
plt.figure(figsize=(6, 3.5))
for name, pts in data.items():
    plt.loglog([p[0] for p in pts], [p[1] for p in pts], 'o-', label=name)
plt.xlabel('vertices n'); plt.ylabel('mean cover time'); plt.legend(); plt.tight_layout(); plt.show()

Every measured cover time is below $2m(n-1)$, as the theorem of Aleliunas, Karp, Lipton, Lovász and Rackoff guarantees for connected graphs. The shapes differ a great deal: about $n\ln n$ steps for the complete graph (a coupon collector), about $n^2$ for paths and cycles (on average exactly $(n-1)^2$ from an end of a path, and $n(n-1)/2$ on a cycle), and about $n^3$ for the lollipop, whose walk keeps falling back into the clique; the bound $2m(n-1)$ is of order $n^3$ in general, and the lollipop shows it cannot be improved by much.

So a randomised machine decides undirected reachability with $O(\log n)$ bits: walk for $4m(n-1)$ steps, and if $t$ was not met, say "no"; by Markov's inequality it errs with probability at most $1/2$, and only on "yes" instances. In 2005 Reingold removed the coins: undirected reachability is in $\mathsf L$.